In [ ]:
EXPECTED_PARROT_API_KEY = 'your-api-key'

In [4]:
from edsl import QuestionFreeText, QuestionNumerical, QuestionDict, QuestionList, Survey, Scenario, ScenarioList, Agent, AgentList, Model, ModelList, FileStore, Coop

Posting the file to Coop:

In [5]:
fs = FileStore('persona.csv')

if refresh := True:
    fs.push(
        description = 'Evidence on Inflation Expectations Formation Using Large Language Models: AI Personas',
        alias = 'inflation-expectation-survey-ai-personas',
        visibility = 'public' # it can be 'public' or 'unlisted'
    )
else:
    fs.patch('https://www.expectedparrot.com/content/alizarif/inflation-expectation-survey-ai-personas', value = fs)

Retrieving the file:

In [6]:
fs = FileStore.pull('https://www.expectedparrot.com/content/alizarif/inflation-expectation-survey-ai-personas')

Creating scenarios for the data:

In [7]:
# Load the CSV file into a pandas DataFrame
import pandas as pd

temp_file_path = fs.to_tempfile()
df = pd.read_csv(temp_file_path)

# Convert the DataFrame into a ScenarioList
demographics_scenarios = ScenarioList([
	Scenario(row.to_dict()) for _, row in df.iterrows()
])

Inspecting the scenarios:

In [10]:
demographics_scenarios[0]

,key,value
0,userid,70111962
1,date,202001
2,Age,49
3,Gender,2
4,Marital,2
5,STATE,VA
6,Education,College
7,Income,Over 100k


In [11]:
demographics_scenarios = demographics_scenarios.rename({
    "userid":"userid",
    "Age":"age",
    "Gender":"gender",
    "Marital":"marital",
    "STATE":"state",
    "Education":"education",
    "Income":"income"
})

In [ ]:
def add_text_fields(scenarios):
    """
    Convert numeric demographic codes to text fields and add full state names to scenarios.
    
    Args:
        scenarios: ScenarioList containing scenarios with demographic data
        
    Returns:
        ScenarioList: New ScenarioList with added text fields (gender_text, marital_text, state_name)
    """
    new_scenarios = None
    
    # Maps for gender and marital status
    gender_map = {
        1: 'female',
        2: 'male'
    }
    
    marital_map = {
        1: 'married',
        2: 'single'
    }
    
    # Map of state codes to full state names
    state_map = {
        'AL': 'Alabama',
        'AK': 'Alaska',
        'AZ': 'Arizona',
        'AR': 'Arkansas',
        'CA': 'California',
        'CO': 'Colorado',
        'CT': 'Connecticut',
        'DE': 'Delaware',
        'FL': 'Florida',
        'GA': 'Georgia',
        'HI': 'Hawaii',
        'ID': 'Idaho',
        'IL': 'Illinois',
        'IN': 'Indiana',
        'IA': 'Iowa',
        'KS': 'Kansas',
        'KY': 'Kentucky',
        'LA': 'Louisiana',
        'ME': 'Maine',
        'MD': 'Maryland',
        'MA': 'Massachusetts',
        'MI': 'Michigan',
        'MN': 'Minnesota',
        'MS': 'Mississippi',
        'MO': 'Missouri',
        'MT': 'Montana',
        'NE': 'Nebraska',
        'NV': 'Nevada',
        'NH': 'New Hampshire',
        'NJ': 'New Jersey',
        'NM': 'New Mexico',
        'NY': 'New York',
        'NC': 'North Carolina',
        'ND': 'North Dakota',
        'OH': 'Ohio',
        'OK': 'Oklahoma',
        'OR': 'Oregon',
        'PA': 'Pennsylvania',
        'RI': 'Rhode Island',
        'SC': 'South Carolina',
        'SD': 'South Dakota',
        'TN': 'Tennessee',
        'TX': 'Texas',
        'UT': 'Utah',
        'VT': 'Vermont',
        'VA': 'Virginia',
        'WA': 'Washington',
        'WV': 'West Virginia',
        'WI': 'Wisconsin',
        'WY': 'Wyoming',
        'DC': 'District of Columbia',
        'PR': 'Puerto Rico',
        'VI': 'Virgin Islands',
        'GU': 'Guam',
        'AS': 'American Samoa',
        'MP': 'Northern Mariana Islands'
    }
    
    # First, convert numeric columns with .000000 format to integers
    numeric_columns = ['gender', 'marital', 'age']
    
    for scenario in scenarios:
        # Convert numeric columns to integers if they're float or string representations of float
        for col in numeric_columns:
            if col in scenario:
                # Handle string representations (e.g., "2.000000")
                if isinstance(scenario[col], str):
                    try:
                        # Convert to float first, then to int if it's a whole number
                        value = float(scenario[col])
                        if value.is_integer():
                            scenario[col] = int(value)
                    except ValueError:
                        pass  # Keep as string if conversion fails
                # Handle float values (e.g., 2.000000)
                elif isinstance(scenario[col], float):
                    if scenario[col].is_integer():
                        scenario[col] = int(scenario[col])
        
        # Now map the integer values to text
        gender = scenario['gender']
        scenario['gender_text'] = gender_map.get(gender, 'other')
        
        marital = scenario['marital']
        scenario['marital_text'] = marital_map.get(marital, 'other')
        
        # Add state full name
        if 'state' in scenario and scenario['state'] in state_map:
            scenario['state_name'] = state_map.get(scenario['state'])
        else:
            scenario['state_name'] = 'Unknown'

        if new_scenarios is None:
            new_scenarios = ScenarioList([scenario])
        else:
            new_scenarios.append(scenario)
            
    return ScenarioList(new_scenarios)

In [34]:
demographics_scenarios = add_text_fields(demographics_scenarios)

In [94]:
demographics_scenarios[0]

,key,value
0,userid,70111962
1,date,202001
2,age,49
3,gender,2
4,marital,2
5,state,VA
6,education,College
7,income,Over 100k
8,gender_text,male
9,marital_text,single


Convert to agent list:

In [37]:
agents = demographics_scenarios.to_agent_list()

In [ ]:
def add_traits_presentation(agentlist):
    """
    Add demographic presentation template to agents in an AgentList.
    
    Args:
        agentlist: AgentList containing agents to modify
        
    Returns:
        AgentList: New AgentList with agents containing traits presentation template
    """
    new_agents = None

    for a in agentlist:
        a = Agent(
            traits = a.traits,
            traits_presentation_template = """
            You are a {{ age }} year old {{ gender_text }} who is {{ marital_text }} 
            with an education level of {{ education }} degree and income category of {{ income }} 
            who lives in the state of {{ state_name }}.
            """
        )

        if new_agents is None:
            new_agents = [a]
        else:
            new_agents.append(a)
            
    return AgentList(new_agents)

In [54]:
agents = add_traits_presentation(agents)

In [55]:
agents[0]

,key,value
0,traits:userid,70111962
1,traits:date,202001
2,traits:age,49
3,traits:gender,2
4,traits:marital,2
5,traits:state,VA
6,traits:education,College
7,traits:income,Over 100k
8,traits:gender_text,male
9,traits:marital_text,single


For simplicity in the EDSL I create 10 groups for each treatment (or scenarios)

In [57]:
import random

# Create 10 random agent groups
num_groups = 10
total_agents = len(agents)
base_agents_per_group = total_agents // num_groups
remaining_agents = total_agents % num_groups

agent_groups = {}

indices = list(range(total_agents))
random.shuffle(indices)

extra_agent_groups = random.sample(range(num_groups), remaining_agents)

current_idx = 0

for i in range(num_groups):
    group_size = base_agents_per_group + (1 if i in extra_agent_groups else 0)
    
    group_indices = indices[current_idx:current_idx + group_size]
    current_idx += group_size
    
    agent_groups[f"agents{i}"] = AgentList([agents[idx] for idx in group_indices])

print(f"Created {num_groups} agent groups with even distribution")
for group_name, group in agent_groups.items():
    print(f"{group_name}: {len(group)} agents")

Created 10 agent groups with even distribution
agents0: 758 agents
agents1: 758 agents
agents2: 758 agents
agents3: 758 agents
agents4: 758 agents
agents5: 758 agents
agents6: 758 agents
agents7: 758 agents
agents8: 758 agents
agents9: 758 agents


In [206]:
from edsl import Agent, AgentList, Coop

# Push each agent group to Coop
for group_name, agent_list in agent_groups.items():
    agent_list.push(
        description=f"Agent group {group_name}",
        alias=f"{group_name}",
        visibility="public"  # Change to "private" or "unlisted" if needed
    )

print("All agent groups successfully pushed to Coop")

All agent groups successfully pushed to Coop


We can inspect the agents in each group

In [58]:
# Create variables in the global namespace for each agent group
for group_name, group in agent_groups.items():
    globals()[group_name] = group

agents2  # check each group

AgentList([Agent(traits = {'userid': 75011962, 'date': 202303, 'age': 76, 'gender': 1, 'marital': 2, 'state': 'IN', 'education': 'Some College', 'income': 'Under 50k', 'gender_text': 'female', 'marital_text': 'single', 'state_name': 'Indiana'}, traits_presentation_template = """
            You are a {{ age }} year old {{ gender_text }} who is {{ marital_text }} 
            with an education level of {{ education }} degree and income category of {{ income }} 
            who lives in the state of {{ state_name }}.
            """), Agent(traits = {'userid': 75012807, 'date': 202306, 'age': 76, 'gender': 2, 'marital': 1, 'state': 'MI', 'education': 'College', 'income': 'Under 50k', 'gender_text': 'male', 'marital_text': 'married', 'state_name': 'Michigan'}, traits_presentation_template = """
            You are a {{ age }} year old {{ gender_text }} who is {{ marital_text }} 
            with an education level of {{ education }} degree and income category of {{ income }} 
            who lives in the state of {{ state_name }}.
            """), Agent(traits = {'userid': 70113848, 'date': 202003, 'age': 75, 'gender': 2, 'marital': 2, 'state': 'SD', 'education': 'Some College', 'income': 'Under 50k', 'gender_text': 'male', 'marital_text': 'single', 'state_name': 'South Dakota'}, traits_presentation_template = """
            You are a {{ age }} year old {{ gender_text }} who is {{ marital_text }} 
            with an education level of {{ education }} degree and income category of {{ income }} 
            who lives in the state of {{ state_name }}.
            """), Agent(traits = {'userid': 70117765, 'date': 202007, 'age': 69, 'gender': 2, 'marital': 2, 'state': 'GA', 'education': 'College', 'income': '50k to 100k', 'gender_text': 'male', 'marital_text': 'single', 'state_name': 'Georgia'}, traits_presentation_template = """
            You are a {{ age }} year old {{ gender_text }} who is {{ marital_text }} 
            with an education level of {{ education }} degree and income category of {{ income }} 
            who lives in the state of {{ state_name }}.
            """), Agent(traits = {'userid': 75015361, 'date': 202310, 'age': 42, 'gender': 1, 'marital': 2, 'state': 'PA', 'education': 'College', 'income': '50k to 100k', 'gender_text': 'female', 'marital_text': 'single', 'state_name': 'Pennsylvania'}, traits_presentation_template = """
            You are a {{ age }} year old {{ gender_text }} who is {{ marital_text }} 
            with an education level of {{ education }} degree and income category of {{ income }} 
            who lives in the state of {{ state_name }}.
            """), Agent(traits = {'userid': 75009613, 'date': 202211, 'age': 30, 'gender': 1, 'marital': 2, 'state': 'PA', 'education': 'Some College', 'income': '50k to 100k', 'gender_text': 'female', 'marital_text': 'single', 'state_name': 'Pennsylvania'}, traits_presentation_template = """
            You are a {{ age }} year old {{ gender_text }} who is {{ marital_text }} 
            with an education level of {{ education }} degree and income category of {{ income }} 
            who lives in the state of {{ state_name }}.
            """), Agent(traits = {'userid': 70125838, 'date': 202105, 'age': 70, 'gender': 1, 'marital': 2, 'state': 'TX', 'education': 'College', 'income': 'Under 50k', 'gender_text': 'female', 'marital_text': 'single', 'state_name': 'Texas'}, traits_presentation_template = """
            You are a {{ age }} year old {{ gender_text }} who is {{ marital_text }} 
            with an education level of {{ education }} degree and income category of {{ income }} 
            who lives in the state of {{ state_name }}.
            """), Agent(traits = {'userid': 70118564, 'date': 202009, 'age': 58, 'gender': 1, 'marital': 2, 'state': 'IL', 'education': 'Some College', 'income': 'Under 50k', 'gender_text': 'female', 'marital_text': 'single', 'state_name': 'Illinois'}, traits_presentation_template = """
            You are a {{ age

In [ ]:
agents2[2]

,key,value
0,traits:userid,70121099
1,traits:date,202012
2,traits:age,65
3,traits:gender,2
4,traits:marital,1
5,traits:state,PA
6,traits:education,College
7,traits:income,Under 50k
8,traits:gender_text,male
9,traits:marital_text,married


Now we create scenarios. The number of scenarios is the same of the number of treatments. Here I have one control group and 9 treatment groups. In total 10 scenarios.

In [61]:
scenarios0 = ScenarioList([
    Scenario({
        "treatment": "T_0", 
        "description": "Control with no information",
        "info": ""
    })
])

scenarios1 = ScenarioList([
    Scenario({
        "treatment": "T_1", 
        "description": "Placebo group",
        "info": "Population of the U.S. grew by 1% between 2022 and 2024."
    })
])

scenarios2 = ScenarioList([
    Scenario({
        "treatment": "T_2", 
        "description": "Current rate, FFR",
        "info": "The interest rate set by the Federal Reserve, known as the Federal Funds Rate, is currently at 4.25%-4.5% range."
    })
])

scenarios3 = ScenarioList([
    Scenario({
        "treatment": "T_3", 
        "description": "Current rate, FFR+ Projection Next Year",
        "info": "The interest rate set by the Federal Reserve, known as the Federal Funds Rate, is currently at 4.25%-4.5% range. One forecast from the Federal Reserve is that this interest rate will be 3.9% on average in 2025"
    })
])

scenarios4 = ScenarioList([
    Scenario({
        "treatment": "T_4", 
        "description": "Current rate, FFR+ Projection Next Years and Longer Run",
        "info": "The interest rate set by the Federal Reserve, known as the Federal Funds Rate, is currently at 4.25%-4.5% range. One forecast from the Federal Reserve is that this interest rate will be 3.9% on average in 2025, 3.4% in 2026, 3.1% in 2027, and 3% in the longer run"
    })
])

scenarios5 = ScenarioList([
    Scenario({
        "treatment": "T_5", 
        "description": "Inflation treatment: current inflation + Past Inflation",
        "info": "Over the last three years, the overall inflation rate in the economy as measured by the percentage change in a consumer price index has been 4.5%."
    })
])

scenarios6 = ScenarioList([
    Scenario({
        "treatment": "T_6", 
        "description": "Inflation treatment: current inflation ",
        "info": "Over the last twelve months, the overall inflation rate in the economy as measured by the percentage change in a consumer price index has been 2.8%."
    })
])

scenarios7 = ScenarioList([
    Scenario({
        "treatment": "T_7", 
        "description": "Inflation treatment: current inflation + Next Year ",
        "info": "Over the last twelve months, the overall inflation rate in the economy as measured by the percentage change in a consumer price index has been 2.8%. One forecast at the Federal Reserve is that this inflation rate will be 2.7% on average in 2025 "
    })
])

scenarios8 = ScenarioList([
    Scenario({
        "treatment": "T_8", 
        "description": "Inflation treatment: current inflation + Next years and the longer run",
        "info": "Over the last twelve months, the overall inflation rate in the economy as measured by the percentage change in a consumer price index has been 2.8%. One forecast at the Federal Reserve is that this inflation rate will be 2.7% on average in 2025, 2.2% in 2026, 2% in 2027, and 2% in the longer run"
    })
])

scenarios9 = ScenarioList([
    Scenario({
        "treatment": "T_9", 
        "description": "Current fixed-rate 30-year mortgage",
        "info": "The current average rate for fixed-rate 30-year mortgage is 6.64% per year."
    })
])

Questions Prior to the information treatment are density forecast

In [62]:
q1_list = QuestionList(
    question_name = "Q1_S_Before_list",
    question_text = """
    Please estimate the probability (as a percentage) for each of the following inflation/deflation scenarios over the next 12 months.
    Each probability must be between 0% and 100%.
    You may use up to 2 decimal points (e.g., 7.25%).
    The sum of all probabilities must equal exactly 100%.
    Return only a list of the numbers (i.e., 50 instead of '50%').
    
    Inflation of 12% or more: _____%
    Inflation between 8% and 12%: _____%
    Inflation between 4% and 8%: _____%
    Inflation between 2% and 4%: _____%
    Inflation between 0% and 2%: _____%
    Deflation between 0% and 2%: _____%
    Deflation between 2% and 4%: _____%
    Deflation between 4% and 8%: _____%
    Deflation between 8% and 12%: _____%
    Deflation of 12% or more: _____%
    """,
)

q2_list = QuestionList(
    question_name = "Q2_L_Before_list",
    question_text = """
    Please estimate the probabiLlity (as a percentage) for each of the following inflation/deflation scenarios over the 
    12-month period beginning 24 months from now and ending 36 months from now.
    Each probability must be between 0% and 100%.
    You may use up to 2 decimal points (e.g., 7.25%).
    The sum of all probabilities must equal exactly 100%.
    Return only a list of the numbers (i.e., 50 instead of '50%').

    Inflation of 12% or more: _____%
    Inflation between 8% and 12%: _____%
    Inflation between 4% and 8%: _____%
    Inflation between 2% and 4%: _____%
    Inflation between 0% and 2%: _____%
    Deflation between 0% and 2%: _____%
    Deflation between 2% and 4%: _____%
    Deflation between 4% and 8%: _____%
    Deflation between 8% and 12%: _____%
    Deflation of 12% or more: _____%
    """,
)

Questions after the treatment are the point forecasts

In [63]:
q3 = QuestionNumerical(
    question_name = "Q1_S_After",
    question_text = """
    Consider the following current event: {{ info }}
    What do you expect the rate of inflation to be over the next 12 months? Please give your best guess. 
    """,
    # min_value = 1
    # max_value = 100 
)

q4 = QuestionNumerical(
    question_name = "Q2_L_After",
    question_text = """
    Consider the following current event: {{ info }}
    What do you expect the rate of inflation to be over the 12-month period beginning 24 months from now
    and ending 36 months from now? 
    Please give your best guess.
    """,
    # min_value = 1, 
    # max_value = 100 
)

Creating the survey with full memory

In [64]:
survey = Survey([q1_list, q2_list, q3, q4])
survey = survey.set_full_memory_mode()

The models of interest can be listed here. 

In [ ]:
models = ModelList([
    #Model("gpt-4o", service_name = "openai", temperature = 1),
    #Model("gpt-4o", service_name = "openai"),
    #Model("gpt-4o", service_name = "openai", temperature = 1.5),
    #Model("gpt-4.1", service_name = "openai", temperature = 1),
    Model("claude-3-5-haiku-20241022", service_name = "anthropic"),
    #Model("claude-3-7-sonnet-20250219", service_name = "anthropic"),
    #Model("gpt-4o-mini", service_name = "openai", temperature = 1),
    #Model("meta-llama/Meta-Llama-3-70B-Instruct", service_name = "deep_infra"),
    #Model("deepseek-ai/DeepSeek-V3", service_name = "deep_infra", temperature = 1),
])

In [263]:
models

,model,inference_service,temperature,max_tokens,top_p,frequency_penalty,presence_penalty,logprobs,top_logprobs
0,claude-3-5-haiku-20241022,anthropic,0.500000,1000,1,0,0,False,3


### Run Survey for each agent group corresponding to the treatment (Scenario) and then save the results. 

#### Some models are better than others in handling API calls. It is better to adjust the batch size for each model.

In [ ]:
import os
import re
import time
from datetime import datetime

# Create lists of all scenario lists and agent lists
all_scenarios = [scenarios0, scenarios1, scenarios2, scenarios3, scenarios4, scenarios5, scenarios6, scenarios7, scenarios8, scenarios9]
all_agents = [agents0, agents1, agents2, agents3, agents4, agents5, agents6, agents7, agents8, agents9]

# Specify which scenario numbers to run
numbers_to_run = [0,1,2,3,4,5,6,7,8,9]  # Example: it can be [0, 1, 2] or any other combination

# Define batch size and wait time between batches (in seconds)
batch_size = 100
wait_time = 10  # 10 seconds

# Get current timestamp in compact format (YYYYMMDD_HHMMSS)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create the main results directory if it doesn't exist
results_dir = "Results"
os.makedirs(results_dir, exist_ok=True)

# Extract model name and temperature from the model string
model_info = str(models)
model_name_match = re.search(r"model_name\s*=\s*'([^']+)'", model_info)
temp_match = re.search(r"temperature\s*=\s*(\d+(?:\.\d+)?)", model_info)
if model_name_match and temp_match:
    model_name = model_name_match.group(1)
    temp = temp_match.group(1)
    folder_name = f"{model_name}(temp={temp})"
else:
    # Fallback if regex doesn't match
    folder_name = "unknown_model"

# Create a subdirectory for the current model
model_dir = os.path.join(results_dir, folder_name)
os.makedirs(model_dir, exist_ok=True)

# Loop through only the selected scenario numbers
for i in numbers_to_run:
    # Make sure the index is valid
    if i < len(all_scenarios) and i < len(all_agents):
        # Get the corresponding scenario list
        scenario_list = all_scenarios[i]
        
        # Calculate the total number of agents
        total_agents = len(all_agents[i])
        
        # Process agents in batches
        for start_idx in range(0, total_agents, batch_size):
            # Calculate end index for this batch
            end_idx = min(start_idx + batch_size - 1, total_agents - 1)
            
            # Get the batch of agents
            agent_list = all_agents[i][start_idx:end_idx + 1]
            
            # Create a descriptive filename with timestamp, scenario number, and agent range
            file_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = os.path.join(model_dir, f"results_{file_timestamp}_scenario{i}_agents{start_idx}-{end_idx}.csv")
            
            # Run the survey with this batch
            print(f"Starting survey for scenario {i}, agents {start_idx}-{end_idx}...")
            results = survey.by(scenario_list).by(agent_list).by(models).run()
            
            # Save to CSV
            results.to_csv(filename)
            
            # Print progress
            print(f"Completed survey for scenario {i}, agents {start_idx}-{end_idx}, saved to {filename}")
            
            # Check if there are more batches to process
            if end_idx < total_agents - 1:
                # Calculate remaining time in a readable format
                minutes = wait_time // 60
                seconds = wait_time % 60
                print(f"Waiting for {minutes} minutes and {seconds} seconds before processing the next batch...")
                
                # Add countdown for wait time
                for remaining in range(wait_time, 0, -10):  # Update every 10 seconds
                    mins = remaining // 60
                    secs = remaining % 60
                    print(f"Next batch starts in: {mins:02d}:{secs:02d}", end="\r")
                    time.sleep(10)
                
                print("\nResuming execution...")
    else:
        print(f"Skipping scenario {i}: Index out of range")

print("All batches have been processed!")

We can post the notebook to the EDSL platform:

In [ ]:
from edsl import Notebook

nb = Notebook("main survey.ipynb")
nb.push(
    description = "Inflation Expectations Formation Using Large Language Models: EDSL code",
    alias = "inflation-expectation-survey-edsl-code",
    visibility = "public" # it can be 'public' or 'unlisted'
)